# TripB Dataset Full Pipeline Notebook

이 노트북은 TripB 전기차 배터리 데이터를 다음 순서로 자동 생성합니다.

1. **Raw Dataset**  
   - 원본 CSV를 `01_raw_individual/`, `01_raw_combined/`에 저장

2. **Preprocessed Dataset**  
   - 센서 컬럼 정리, 결측치 제거, 기본 변화량/BSI/라벨 생성  
   - `02_preprocessed_individual/`, `02_preprocessed_combined/`

3. **Augmented Dataset**  
   - Train 데이터에만 Gaussian Noise 기반 증강 적용  
   - `03_augmented_individual/`, `03_augmented_combined/`

4. **Preprocessed + Augmented Combined Dataset**  
   - 전처리 데이터와 증강 데이터를 합친 학습 후보 데이터  
   - `04_preprocessed_plus_augmented/`

5. **Feature Dataset**  
   - 5분 Window Aggregation 파생변수 생성  
   - `05_feature_individual/`, `05_feature_combined/`

6. **Sampled Dataset**  
   - 최종 학습용 샘플링 데이터 생성  
   - `06_sampled_individual/`, `06_sampled_combined/`

> 핵심 원칙: 증강은 `voltage`, `current`, `battery_temp`, `ambient_temp` 같은 센서 원본 계열에만 적용하고, BSI, Z_BSI, label, window feature는 증강된 센서값을 기준으로 다시 계산합니다.

In [ ]:
# ============================================================
# 0. 라이브러리 및 기본 설정
# ============================================================

import os
import glob
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)

# ============================================================
# 사용자가 수정할 경로
# ============================================================

# 원본 TripB CSV가 들어 있는 폴더
RAW_INPUT_DIR = r"C:\Users\EL095\Microsoft\20260515\Trip Data(Original)"

# 결과물이 저장될 최상위 폴더
PROJECT_OUTPUT_DIR = r"C:\Users\EL095\Microsoft\20260515\TripB_Full_Dataset_Pipeline"

# 파일명 필터
FILE_PATTERN = "*TripB*.csv"

# ============================================================
# 주요 파라미터
# ============================================================

THERMAL_WINDOW_MIN = 70
WINDOW_MIN = 5

# 증강 설정
AUGMENT_TRAIN_ONLY = True
N_AUGMENTED_FILES = 10
START_AUG_VEHICLE_NUM = 39
NOISE_LEVEL = 0.01
RANDOM_SEED = 42

# 샘플링 설정
TARGET_TOTAL_N = 60000
KEEP_AROUND_MIN = 5
DANGER_RATIO = 0.05
WARNING_RATIO = 0.05
NEAR_NORMAL_RATIO = 0.45

rng = np.random.default_rng(RANDOM_SEED)

print("RAW_INPUT_DIR:", RAW_INPUT_DIR)
print("PROJECT_OUTPUT_DIR:", PROJECT_OUTPUT_DIR)
print("FILE_PATTERN:", FILE_PATTERN)

In [ ]:
# ============================================================
# 1. 출력 폴더 구조 생성
# ============================================================

BASE_DIR = Path(PROJECT_OUTPUT_DIR)

DIRS = {
    "raw_individual": BASE_DIR / "01_raw_individual",
    "raw_combined": BASE_DIR / "01_raw_combined",
    "preprocessed_individual": BASE_DIR / "02_preprocessed_individual",
    "preprocessed_combined": BASE_DIR / "02_preprocessed_combined",
    "augmented_individual": BASE_DIR / "03_augmented_individual",
    "augmented_combined": BASE_DIR / "03_augmented_combined",
    "preprocessed_plus_augmented": BASE_DIR / "04_preprocessed_plus_augmented",
    "feature_individual": BASE_DIR / "05_feature_individual",
    "feature_combined": BASE_DIR / "05_feature_combined",
    "sampled_individual": BASE_DIR / "06_sampled_individual",
    "sampled_combined": BASE_DIR / "06_sampled_combined",
    "logs": BASE_DIR / "logs",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Created folders:")
for name, path in DIRS.items():
    print(f"{name:32s} -> {path}")

In [ ]:
# ============================================================
# 2. 공통 유틸 함수
# ============================================================

def find_col(df, candidates, keyword_contains=None):
    # 후보 컬럼명 또는 키워드 포함 여부로 컬럼을 찾는 함수
    for col in candidates:
        if col in df.columns:
            return col

    if keyword_contains is not None:
        for col in df.columns:
            col_str = str(col)
            if all(kw in col_str for kw in keyword_contains):
                return col

    return None


def zscore(series):
    # 일반 Z-score. 표준편차가 0이면 0으로 처리
    mean = series.mean()
    std = series.std()

    if std == 0 or pd.isna(std):
        return pd.Series(0, index=series.index)

    return (series - mean) / std


def calc_slope(values):
    # rolling window 내부 기울기 계산
    values = np.asarray(values)

    if len(values) < 2:
        return 0

    x = np.arange(len(values))

    try:
        return np.polyfit(x, values, 1)[0]
    except Exception:
        return 0


def clean_status_name(df):
    # Normal 라벨 표기 통일
    df = df.copy()

    if "current_status_label" in df.columns and "status_name" in df.columns:
        df.loc[df["current_status_label"] == 0, "status_name"] = "Normal"
        df.loc[df["status_name"] == "Unknown", "status_name"] = "Normal"

    return df


def save_combined(dfs, output_path, sort_cols=None):
    # 여러 DataFrame을 하나로 합쳐 저장
    if len(dfs) == 0:
        raise ValueError("합칠 DataFrame이 없습니다.")

    combined = pd.concat(dfs, ignore_index=True)

    if sort_cols is not None:
        existing_sort_cols = [c for c in sort_cols if c in combined.columns]
        if existing_sort_cols:
            combined = combined.sort_values(existing_sort_cols).reset_index(drop=True)

    combined.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("Saved combined:", output_path)
    print("Shape:", combined.shape)

    if "status_name" in combined.columns:
        print(combined["status_name"].value_counts())

    return combined


def print_status_summary(df, title="Status Summary"):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print("Rows:", len(df))

    if "status_name" in df.columns:
        result = pd.DataFrame({
            "count": df["status_name"].value_counts(),
            "ratio(%)": df["status_name"].value_counts(normalize=True).mul(100).round(2)
        })
        display(result)

    if "current_status_label" in df.columns:
        print("\nLabel distribution:")
        print(df["current_status_label"].value_counts().sort_index())

In [ ]:
# ============================================================
# 3. Raw Dataset 생성
#    - 원본 파일 복사
#    - 원본 combined 생성
# ============================================================

raw_files = sorted(glob.glob(os.path.join(RAW_INPUT_DIR, FILE_PATTERN)))

print("Raw TripB file count:", len(raw_files))

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"원본 파일을 찾지 못했습니다. RAW_INPUT_DIR와 FILE_PATTERN을 확인하세요.\n"
        f"RAW_INPUT_DIR={RAW_INPUT_DIR}\nFILE_PATTERN={FILE_PATTERN}"
    )

raw_dfs = []
raw_info = []

for idx, file_path in enumerate(raw_files, start=1):
    file_name = os.path.basename(file_path)
    vehicle_id = f"VehicleB_{idx:03d}"

    # 1) 원본 파일 그대로 복사
    copied_path = DIRS["raw_individual"] / f"{vehicle_id}_{file_name}"
    shutil.copy2(file_path, copied_path)

    # 2) combined 생성을 위해 읽기
    raw_df = pd.read_csv(file_path, sep=";", encoding="latin1", low_memory=False)
    raw_df["vehicle_id"] = vehicle_id
    raw_df["source"] = file_name

    raw_dfs.append(raw_df)
    raw_info.append({
        "vehicle_id": vehicle_id,
        "file_name": file_name,
        "rows": len(raw_df),
        "copied_path": str(copied_path)
    })

raw_info_df = pd.DataFrame(raw_info)
raw_info_path = DIRS["logs"] / "raw_file_info.csv"
raw_info_df.to_csv(raw_info_path, index=False, encoding="utf-8-sig")

raw_combined_path = DIRS["raw_combined"] / "TripB_raw_combined.csv"
raw_combined_df = save_combined(raw_dfs, raw_combined_path, sort_cols=["vehicle_id"])

print("\nRaw file info saved:", raw_info_path)
display(raw_info_df)

In [ ]:
# ============================================================
# 4. 전처리 함수
#    Raw -> Preprocessed
# ============================================================

def preprocess_single_tripb_file(file_path, vehicle_id):
    file_name = os.path.basename(file_path)

    raw_df = pd.read_csv(
        file_path,
        sep=";",
        encoding="latin1",
        low_memory=False
    )

    raw_df["source"] = file_name
    raw_df["vehicle_id"] = vehicle_id

    time_col = find_col(raw_df, ["Time [s]", "Time[s]", "time"], ["Time"])
    voltage_col = find_col(raw_df, ["Battery Voltage [V]", "Battery Voltage"], ["Battery Voltage"])
    current_col = find_col(raw_df, ["Battery Current [A]", "Battery Current"], ["Battery Current"])

    battery_temp_col = find_col(
        raw_df,
        [
            "Battery Temperature [°C]",
            "Battery Temperature [�C]",
            "Battery Temperature [Â°C]",
            "Battery Temperature"
        ],
        ["Battery Temperature"]
    )

    ambient_temp_col = find_col(
        raw_df,
        [
            "Ambient Temperature [°C]",
            "Ambient Temperature [�C]",
            "Ambient Temperature [Â°C]",
            "Ambient Temperature"
        ],
        ["Ambient Temperature"]
    )

    required_cols = {
        "vehicle_id": "vehicle_id",
        "source": "source",
        time_col: "time",
        voltage_col: "voltage",
        current_col: "current",
        battery_temp_col: "battery_temp",
        ambient_temp_col: "ambient_temp",
    }

    if any(k is None for k in required_cols.keys()):
        print("컬럼 탐색 실패:", file_name)
        print(raw_df.columns.tolist())
        raise KeyError(f"{file_name}에서 필수 컬럼을 찾지 못했습니다.")

    df = raw_df[list(required_cols.keys())].rename(columns=required_cols).copy()

    numeric_cols = [
        "time",
        "voltage",
        "current",
        "battery_temp",
        "ambient_temp"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=numeric_cols).copy()
    dropped = before - len(df)

    df = df.sort_values("time").reset_index(drop=True)

    df["delta_t"] = df["time"].diff().replace(0, np.nan)
    median_dt = df["delta_t"].median()

    if pd.isna(median_dt) or median_dt <= 0:
        median_dt = 0.1

    df["delta_t"] = df["delta_t"].fillna(median_dt)

    df["delta_v"] = df["voltage"].diff().fillna(0)
    df["delta_i"] = df["current"].diff().fillna(0)
    df["delta_t_temp"] = df["battery_temp"].diff().fillna(0)

    df["power"] = df["voltage"] * df["current"]
    df["abs_power"] = df["power"].abs()
    df["delta_p"] = df["power"].diff().fillna(0)

    df["dv_dt"] = df["delta_v"] / df["delta_t"]
    df["di_dt"] = df["delta_i"] / df["delta_t"]
    df["dt_dt"] = df["delta_t_temp"] / df["delta_t"]
    df["dp_dt"] = df["delta_p"] / df["delta_t"]

    rate_cols = ["dv_dt", "di_dt", "dt_dt", "dp_dt"]
    df[rate_cols] = df[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    df["temp_diff"] = df["battery_temp"] - df["ambient_temp"]
    df["joule_heating_stress"] = df["current"] ** 2

    thermal_window_rows = max(int(round((THERMAL_WINDOW_MIN * 60) / median_dt)), 1)
    df["thermal_temperature_70min"] = (
        df["abs_power"]
        .rolling(window=thermal_window_rows, min_periods=1)
        .mean()
    )

    df["rolling_abs_power_70min"] = df["thermal_temperature_70min"]

    df["Z_Delta_I"] = zscore(df["delta_i"])
    df["Z_Delta_V"] = zscore(df["delta_v"])
    df["Z_Battery_Current"] = zscore(df["current"])
    df["Z_Battery_Voltage"] = zscore(df["voltage"])
    df["Z_Joule_Heating_Stress"] = zscore(df["joule_heating_stress"])
    df["Z_Thermal_Temperature_70min"] = zscore(df["thermal_temperature_70min"])

    df["Z_Thermal_Stress"] = (
        0.8 * df["Z_Joule_Heating_Stress"]
        + 0.2 * df["Z_Thermal_Temperature_70min"]
    )

    df["thermal_stress"] = df["Z_Thermal_Stress"]

    df["BSI"] = (
        0.4830 * df["Z_Delta_I"].abs()
        + 0.2218 * df["Z_Delta_V"].abs()
        + 0.1027 * df["Z_Thermal_Stress"].abs()
        + 0.0992 * df["Z_Battery_Current"].abs()
        + 0.0933 * df["Z_Battery_Voltage"].abs()
    )

    df["Z_BSI"] = zscore(df["BSI"])

    df["current_status_label"] = np.select(
        [
            df["Z_BSI"] < 2,
            (df["Z_BSI"] >= 2) & (df["Z_BSI"] < 3),
            df["Z_BSI"] >= 3
        ],
        [0, 1, 2],
        default=np.nan
    )

    df["status_name"] = np.select(
        [
            df["Z_BSI"] < 2,
            (df["Z_BSI"] >= 2) & (df["Z_BSI"] < 3),
            df["Z_BSI"] >= 3
        ],
        ["Normal", "Warning", "Danger"],
        default="Unknown"
    )

    df = clean_status_name(df)

    final_cols = [
        "vehicle_id", "source", "time",
        "voltage", "current", "battery_temp", "ambient_temp",
        "delta_t", "delta_v", "delta_i", "delta_t_temp", "delta_p",
        "power", "abs_power", "dv_dt", "di_dt", "dt_dt", "dp_dt",
        "temp_diff", "joule_heating_stress",
        "thermal_temperature_70min", "rolling_abs_power_70min",
        "Z_Delta_I", "Z_Delta_V", "Z_Battery_Current", "Z_Battery_Voltage",
        "Z_Joule_Heating_Stress", "Z_Thermal_Temperature_70min",
        "Z_Thermal_Stress", "thermal_stress",
        "BSI", "Z_BSI",
        "current_status_label", "status_name"
    ]

    existing_final_cols = [c for c in final_cols if c in df.columns]
    df = df[existing_final_cols].copy()

    return df, dropped

In [ ]:
# ============================================================
# 5. Preprocessed Dataset 생성
#    개별 파일 + combined 파일
# ============================================================

preprocessed_dfs = []
preprocess_logs = []

for idx, file_path in enumerate(raw_files, start=1):
    vehicle_id = f"VehicleB_{idx:03d}"
    file_name = os.path.basename(file_path)
    base_name = os.path.splitext(file_name)[0]

    print("\nProcessing:", file_name)

    processed_df, dropped_rows = preprocess_single_tripb_file(file_path, vehicle_id)

    output_name = f"{vehicle_id}_{base_name}_preprocessed.csv"
    output_path = DIRS["preprocessed_individual"] / output_name

    processed_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    preprocessed_dfs.append(processed_df)

    preprocess_logs.append({
        "vehicle_id": vehicle_id,
        "file_name": file_name,
        "output_name": output_name,
        "rows": len(processed_df),
        "dropped_rows": dropped_rows
    })

    print("Saved:", output_path)
    print("Shape:", processed_df.shape)
    print(processed_df["status_name"].value_counts())

preprocess_log_df = pd.DataFrame(preprocess_logs)
preprocess_log_path = DIRS["logs"] / "preprocess_log.csv"
preprocess_log_df.to_csv(preprocess_log_path, index=False, encoding="utf-8-sig")

preprocessed_combined_path = DIRS["preprocessed_combined"] / "TripB_preprocessed_combined.csv"
preprocessed_combined_df = save_combined(
    preprocessed_dfs,
    preprocessed_combined_path,
    sort_cols=["vehicle_id", "time"]
)

print("\nPreprocess log saved:", preprocess_log_path)
display(preprocess_log_df)
print_status_summary(preprocessed_combined_df, "Preprocessed Combined Summary")

In [ ]:
# ============================================================
# 6. 증강 함수
#    Preprocessed -> Augmented
# ============================================================

def add_gaussian_noise_to_sensor_columns(df, noise_level=0.01, rng=None):
    # 센서 원본 계열 컬럼에만 Gaussian Noise 적용
    # noisy_value = original_value + Normal(0, column_std * noise_level)
    # 적용 컬럼: voltage, current, battery_temp, ambient_temp
    if rng is None:
        rng = np.random.default_rng(42)

    df = df.copy()
    sensor_cols = ["voltage", "current", "battery_temp", "ambient_temp"]

    for col in sensor_cols:
        if col not in df.columns:
            raise KeyError(f"센서 컬럼이 없습니다: {col}")

        col_std = df[col].std()

        if pd.isna(col_std) or col_std == 0:
            noise = 0
        else:
            noise = rng.normal(
                loc=0,
                scale=col_std * noise_level,
                size=len(df)
            )

        df[col] = df[col] + noise

    # 물리적으로 말이 안 되는 전압 방지
    df["voltage"] = df["voltage"].clip(lower=0)

    return df


def recalculate_features_after_sensor_change(df):
    # 증강된 센서값 기준으로 delta, power, thermal stress, BSI, label을 다시 계산.
    # window feature는 뒤의 Feature Dataset 단계에서 한 번에 생성.
    df = df.copy()
    df.columns = df.columns.str.strip()

    required_cols = [
        "vehicle_id", "source", "time",
        "voltage", "current", "battery_temp", "ambient_temp"
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"필수 컬럼이 없습니다: {missing_cols}")

    df = df.sort_values(["vehicle_id", "time"]).reset_index(drop=True)
    result_list = []

    for vehicle_id, g in df.groupby("vehicle_id"):
        g = g.sort_values("time").copy()

        g["delta_t"] = g["time"].diff().replace(0, np.nan)
        median_dt = g["delta_t"].median()

        if pd.isna(median_dt) or median_dt <= 0:
            median_dt = 0.1

        g["delta_t"] = g["delta_t"].fillna(median_dt)

        g["delta_v"] = g["voltage"].diff().fillna(0)
        g["delta_i"] = g["current"].diff().fillna(0)
        g["delta_t_temp"] = g["battery_temp"].diff().fillna(0)

        g["power"] = g["voltage"] * g["current"]
        g["abs_power"] = g["power"].abs()
        g["delta_p"] = g["power"].diff().fillna(0)

        g["dv_dt"] = g["delta_v"] / g["delta_t"]
        g["di_dt"] = g["delta_i"] / g["delta_t"]
        g["dt_dt"] = g["delta_t_temp"] / g["delta_t"]
        g["dp_dt"] = g["delta_p"] / g["delta_t"]

        rate_cols = ["dv_dt", "di_dt", "dt_dt", "dp_dt"]
        g[rate_cols] = g[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

        g["temp_diff"] = g["battery_temp"] - g["ambient_temp"]
        g["joule_heating_stress"] = g["current"] ** 2

        thermal_window_rows = max(int(round((THERMAL_WINDOW_MIN * 60) / median_dt)), 1)

        g["thermal_temperature_70min"] = (
            g["abs_power"]
            .rolling(window=thermal_window_rows, min_periods=1)
            .mean()
        )

        g["rolling_abs_power_70min"] = g["thermal_temperature_70min"]

        g["Z_Delta_I"] = zscore(g["delta_i"])
        g["Z_Delta_V"] = zscore(g["delta_v"])
        g["Z_Battery_Current"] = zscore(g["current"])
        g["Z_Battery_Voltage"] = zscore(g["voltage"])
        g["Z_Joule_Heating_Stress"] = zscore(g["joule_heating_stress"])
        g["Z_Thermal_Temperature_70min"] = zscore(g["thermal_temperature_70min"])

        g["Z_Thermal_Stress"] = (
            0.8 * g["Z_Joule_Heating_Stress"]
            + 0.2 * g["Z_Thermal_Temperature_70min"]
        )

        g["thermal_stress"] = g["Z_Thermal_Stress"]

        g["BSI"] = (
            0.4830 * g["Z_Delta_I"].abs()
            + 0.2218 * g["Z_Delta_V"].abs()
            + 0.1027 * g["Z_Thermal_Stress"].abs()
            + 0.0992 * g["Z_Battery_Current"].abs()
            + 0.0933 * g["Z_Battery_Voltage"].abs()
        )

        g["Z_BSI"] = zscore(g["BSI"])

        g["current_status_label"] = np.select(
            [
                g["Z_BSI"] < 2,
                (g["Z_BSI"] >= 2) & (g["Z_BSI"] < 3),
                g["Z_BSI"] >= 3
            ],
            [0, 1, 2],
            default=np.nan
        )

        g["status_name"] = np.select(
            [
                g["Z_BSI"] < 2,
                (g["Z_BSI"] >= 2) & (g["Z_BSI"] < 3),
                g["Z_BSI"] >= 3
            ],
            ["Normal", "Warning", "Danger"],
            default="Unknown"
        )

        g = clean_status_name(g)
        result_list.append(g)

    return pd.concat(result_list, ignore_index=True)

In [ ]:
# ============================================================
# 7. Augmented Dataset 생성
#    개별 증강 파일 + 증강 combined 파일
# ============================================================

preprocessed_files = sorted(glob.glob(str(DIRS["preprocessed_individual"] / "*_preprocessed.csv")))

print("Preprocessed individual file count:", len(preprocessed_files))

if len(preprocessed_files) == 0:
    raise FileNotFoundError("증강 대상 전처리 개별 파일이 없습니다.")

augmented_dfs = []
augmented_logs = []

for i in range(N_AUGMENTED_FILES):
    new_vehicle_num = START_AUG_VEHICLE_NUM + i
    new_vehicle_id = f"VehicleB_{new_vehicle_num:03d}"
    trip_num = i + 1
    trip_name = f"TripB{trip_num:02d}"

    source_path = preprocessed_files[i % len(preprocessed_files)]
    source_base = os.path.basename(source_path)

    print("\n" + "=" * 60)
    print("Creating augmented file")
    print("New vehicle_id:", new_vehicle_id)
    print("Base source file:", source_base)

    df = pd.read_csv(source_path)
    df.columns = df.columns.str.strip()

    df["vehicle_id"] = new_vehicle_id
    df["source"] = f"{trip_name}_augmented_from_{source_base}"

    noisy_df = add_gaussian_noise_to_sensor_columns(
        df,
        noise_level=NOISE_LEVEL,
        rng=rng
    )

    augmented_df = recalculate_features_after_sensor_change(noisy_df)

    output_name = f"{new_vehicle_id}_{trip_name}_augmented_preprocessed.csv"
    output_path = DIRS["augmented_individual"] / output_name

    augmented_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    augmented_dfs.append(augmented_df)

    augmented_logs.append({
        "new_vehicle_id": new_vehicle_id,
        "trip_name": trip_name,
        "base_source_file": source_base,
        "output_name": output_name,
        "rows": len(augmented_df),
        "noise_level": NOISE_LEVEL
    })

    print("Saved:", output_path)
    print("Shape:", augmented_df.shape)
    print(augmented_df["status_name"].value_counts())

augmented_log_df = pd.DataFrame(augmented_logs)
augmented_log_path = DIRS["logs"] / "augmentation_log.csv"
augmented_log_df.to_csv(augmented_log_path, index=False, encoding="utf-8-sig")

augmented_combined_path = DIRS["augmented_combined"] / "TripB_augmented_preprocessed_combined.csv"
augmented_combined_df = save_combined(
    augmented_dfs,
    augmented_combined_path,
    sort_cols=["vehicle_id", "time"]
)

print("\nAugmentation log saved:", augmented_log_path)
display(augmented_log_df)
print_status_summary(augmented_combined_df, "Augmented Combined Summary")

In [ ]:
# ============================================================
# 8. Preprocessed + Augmented Combined Dataset 생성
# ============================================================

preprocessed_plus_augmented_df = pd.concat(
    [preprocessed_combined_df, augmented_combined_df],
    ignore_index=True
)

preprocessed_plus_augmented_df = preprocessed_plus_augmented_df.sort_values(
    ["vehicle_id", "time"]
).reset_index(drop=True)

preprocessed_plus_augmented_path = (
    DIRS["preprocessed_plus_augmented"] /
    "TripB_preprocessed_plus_augmented_combined.csv"
)

preprocessed_plus_augmented_df.to_csv(
    preprocessed_plus_augmented_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", preprocessed_plus_augmented_path)
print("Shape:", preprocessed_plus_augmented_df.shape)
print_status_summary(
    preprocessed_plus_augmented_df,
    "Preprocessed + Augmented Combined Summary"
)

In [ ]:
# ============================================================
# 9. Feature Dataset 생성 함수
#    Preprocessed + Augmented -> Feature Dataset
# ============================================================

def add_window_features(df, window_min=5):
    df = df.copy()
    df.columns = df.columns.str.strip()

    required_cols = [
        "vehicle_id", "time", "BSI", "Z_BSI",
        "delta_i", "delta_v", "current", "power",
        "battery_temp", "temp_diff",
        "current_status_label", "status_name"
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"필수 컬럼이 없습니다: {missing_cols}")

    df = df.sort_values(["vehicle_id", "time"]).reset_index(drop=True)
    result_list = []

    for vehicle_id, g in df.groupby("vehicle_id"):
        g = g.sort_values("time").copy()

        median_dt = g["time"].diff().median()

        if pd.isna(median_dt) or median_dt <= 0:
            median_dt = g["delta_t"].median() if "delta_t" in g.columns else 0.1

        if pd.isna(median_dt) or median_dt <= 0:
            median_dt = 0.1

        window_rows = max(int(round((window_min * 60) / median_dt)), 2)
        rolling = g.rolling(window=window_rows, min_periods=1)

        g["BSI_mean"] = rolling["BSI"].mean()
        g["BSI_max"] = rolling["BSI"].max()
        g["BSI_min"] = rolling["BSI"].min()
        g["BSI_std"] = rolling["BSI"].std().fillna(0)

        g["Z_BSI_mean"] = rolling["Z_BSI"].mean()
        g["Z_BSI_max"] = rolling["Z_BSI"].max()

        g["delta_i_max_abs"] = (
            g["delta_i"].abs()
            .rolling(window=window_rows, min_periods=1)
            .max()
        )

        g["delta_v_max_abs"] = (
            g["delta_v"].abs()
            .rolling(window=window_rows, min_periods=1)
            .max()
        )

        g["current_max_abs"] = (
            g["current"].abs()
            .rolling(window=window_rows, min_periods=1)
            .max()
        )

        g["power_max_abs"] = (
            g["power"].abs()
            .rolling(window=window_rows, min_periods=1)
            .max()
        )

        g["battery_temp_mean"] = rolling["battery_temp"].mean()
        g["temp_diff_mean"] = rolling["temp_diff"].mean()
        g["temp_diff_max"] = rolling["temp_diff"].max()

        g["BSI_slope"] = (
            g["BSI"]
            .rolling(window=window_rows, min_periods=2)
            .apply(calc_slope, raw=True)
            .fillna(0)
        )

        result_list.append(g)

    return pd.concat(result_list, ignore_index=True)

In [ ]:
# ============================================================
# 10. Feature Dataset 생성
#     개별 파일 + combined 파일
# ============================================================

feature_dfs = []

for vehicle_id, g in preprocessed_plus_augmented_df.groupby("vehicle_id"):
    g = g.sort_values("time").reset_index(drop=True)

    feature_df = add_window_features(g, window_min=WINDOW_MIN)
    feature_df = clean_status_name(feature_df)

    source_value = str(feature_df["source"].iloc[0]) if "source" in feature_df.columns and len(feature_df) > 0 else vehicle_id
    safe_source = (
        source_value
        .replace("\\", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )

    output_name = f"{vehicle_id}_{safe_source}_feature.csv"
    output_path = DIRS["feature_individual"] / output_name

    feature_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    feature_dfs.append(feature_df)

    print("Saved:", output_path, "Shape:", feature_df.shape)

feature_combined_path = DIRS["feature_combined"] / "TripB_feature_combined.csv"
feature_combined_df = save_combined(
    feature_dfs,
    feature_combined_path,
    sort_cols=["vehicle_id", "time"]
)

print_status_summary(feature_combined_df, "Feature Combined Summary")

In [ ]:
# ============================================================
# 11. 샘플링 함수
#     Feature Dataset -> Sampled Dataset
# ============================================================

def time_even_sample(data, n):
    data = data.sort_values("time").copy()

    if n <= 0 or len(data) == 0:
        return data.iloc[0:0].copy()

    if len(data) <= n:
        return data.copy()

    idx = np.linspace(0, len(data) - 1, n).astype(int)
    return data.iloc[idx].copy()


def sample_single_file(df, target_n):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "current_status_label" not in df.columns:
        raise KeyError("current_status_label 컬럼이 없습니다.")

    if "time" not in df.columns:
        raise KeyError("time 컬럼이 없습니다.")

    df = df.sort_values("time").reset_index(drop=True)

    df["is_risk_zone"] = df["current_status_label"].isin([1, 2]).astype(int)

    median_dt = df["delta_t"].median() if "delta_t" in df.columns else df["time"].diff().median()

    if pd.isna(median_dt) or median_dt <= 0:
        median_dt = 0.1

    rows_per_min = max(int(round(60 / median_dt)), 1)
    keep_window_rows = KEEP_AROUND_MIN * rows_per_min

    risk_idx = np.where(df["is_risk_zone"].values == 1)[0]
    keep_mask = np.zeros(len(df), dtype=bool)

    for idx in risk_idx:
        start = max(0, idx - keep_window_rows)
        end = min(len(df), idx + keep_window_rows + 1)
        keep_mask[start:end] = True

    df["is_near_risk_zone"] = keep_mask.astype(int)

    danger_df = df[df["current_status_label"] == 2].copy()
    warning_df = df[df["current_status_label"] == 1].copy()
    near_normal_df = df[
        (df["current_status_label"] == 0)
        & (df["is_near_risk_zone"] == 1)
    ].copy()
    far_normal_df = df[
        (df["current_status_label"] == 0)
        & (df["is_near_risk_zone"] == 0)
    ].copy()

    danger_n = int(target_n * DANGER_RATIO)
    warning_n = int(target_n * WARNING_RATIO)
    near_normal_n = int(target_n * NEAR_NORMAL_RATIO)
    far_normal_n = target_n - danger_n - warning_n - near_normal_n

    sampled = pd.concat(
        [
            time_even_sample(danger_df, danger_n),
            time_even_sample(warning_df, warning_n),
            time_even_sample(near_normal_df, near_normal_n),
            time_even_sample(far_normal_df, far_normal_n),
        ],
        ignore_index=True
    )

    if len(sampled) < target_n:
        need = target_n - len(sampled)
        sampled_index_set = set(sampled.index)
        extra_pool = df.loc[~df.index.isin(sampled_index_set)].copy()

        if len(extra_pool) > 0:
            sampled = pd.concat(
                [sampled, time_even_sample(extra_pool, need)],
                ignore_index=True
            )

    if len(sampled) > target_n:
        sampled = time_even_sample(sampled, target_n)

    sampled = clean_status_name(sampled)
    sampled = sampled.sort_values("time").reset_index(drop=True)

    return sampled

In [ ]:
# ============================================================
# 12. Sampled Dataset 생성
#     개별 파일 + combined 파일
# ============================================================

feature_files = sorted(glob.glob(str(DIRS["feature_individual"] / "*_feature.csv")))

print("Feature individual file count:", len(feature_files))

if len(feature_files) == 0:
    raise FileNotFoundError("샘플링 대상 feature 개별 파일이 없습니다.")

file_infos = []

for path in feature_files:
    temp = pd.read_csv(path)
    temp.columns = temp.columns.str.strip()

    vehicle_id = temp["vehicle_id"].iloc[0] if "vehicle_id" in temp.columns and len(temp) > 0 else os.path.basename(path)

    file_infos.append({
        "path": path,
        "vehicle_id": vehicle_id,
        "file_name": os.path.basename(path),
        "rows": len(temp)
    })

file_info_df = pd.DataFrame(file_infos)

total_rows = file_info_df["rows"].sum()

file_info_df["target_n"] = (
    TARGET_TOTAL_N * file_info_df["rows"] / total_rows
).round().astype(int)

file_info_df["target_n"] = file_info_df["target_n"].clip(lower=1)

diff = TARGET_TOTAL_N - file_info_df["target_n"].sum()

if diff != 0:
    largest_idx = file_info_df["rows"].idxmax()
    file_info_df.loc[largest_idx, "target_n"] += diff

sampling_allocation_path = DIRS["logs"] / "sampling_allocation.csv"
file_info_df.to_csv(sampling_allocation_path, index=False, encoding="utf-8-sig")

print("Sampling allocation:")
display(file_info_df)
print("Target total:", file_info_df["target_n"].sum())
print("Sampling allocation saved:", sampling_allocation_path)

sampled_dfs = []

for _, row in file_info_df.iterrows():
    input_path = row["path"]
    target_n = int(row["target_n"])

    df_file = pd.read_csv(input_path)
    df_file.columns = df_file.columns.str.strip()

    sampled_file = sample_single_file(df_file, target_n)

    base_name = os.path.splitext(os.path.basename(input_path))[0]
    output_path = DIRS["sampled_individual"] / f"{base_name}_sampled.csv"

    sampled_file.to_csv(output_path, index=False, encoding="utf-8-sig")
    sampled_dfs.append(sampled_file)

    print("\nSaved:", output_path)
    print("Target:", target_n, "Actual:", len(sampled_file))
    print(sampled_file["status_name"].value_counts())

sampled_combined_df = pd.concat(sampled_dfs, ignore_index=True)

if len(sampled_combined_df) > TARGET_TOTAL_N:
    sampled_combined_df = sampled_combined_df.sort_values(["vehicle_id", "time"]).reset_index(drop=True)
    idx = np.linspace(0, len(sampled_combined_df) - 1, TARGET_TOTAL_N).astype(int)
    sampled_combined_df = sampled_combined_df.iloc[idx].copy()

sampled_combined_df = clean_status_name(sampled_combined_df)
sampled_combined_df = sampled_combined_df.sort_values(["vehicle_id", "time"]).reset_index(drop=True)

sampled_combined_path = DIRS["sampled_combined"] / "TripB_final_sampled_60000_combined.csv"

sampled_combined_df.to_csv(
    sampled_combined_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("Final sampled combined saved:")
print(sampled_combined_path)
print("Final shape:", sampled_combined_df.shape)
print_status_summary(sampled_combined_df, "Final Sampled Combined Summary")

In [ ]:
# ============================================================
# 13. 최종 컬럼 검증
# ============================================================

final_required_cols = [
    "vehicle_id", "source", "time",
    "voltage", "current", "battery_temp", "ambient_temp",
    "delta_v", "delta_i", "delta_t_temp", "delta_p",
    "power", "dv_dt", "di_dt", "dt_dt", "dp_dt",
    "temp_diff", "BSI", "Z_BSI",
    "BSI_mean", "BSI_max", "BSI_min", "BSI_std", "BSI_slope",
    "Z_BSI_mean", "Z_BSI_max",
    "delta_i_max_abs", "delta_v_max_abs",
    "current_max_abs", "power_max_abs",
    "battery_temp_mean", "temp_diff_mean", "temp_diff_max",
    "current_status_label", "status_name"
]

missing_final_cols = [
    col for col in final_required_cols
    if col not in sampled_combined_df.columns
]

print("=" * 70)
print("Final Column Check")
print("=" * 70)

if missing_final_cols:
    print("누락된 컬럼:")
    print(missing_final_cols)
else:
    print("모든 필수 컬럼이 존재합니다.")

print("\nFinal shape:", sampled_combined_df.shape)
print("\nTarget distribution:")
print(sampled_combined_df["current_status_label"].value_counts().sort_index())

print("\nStatus distribution:")
print(sampled_combined_df["status_name"].value_counts())

display(sampled_combined_df.head())

In [ ]:
# ============================================================
# 14. 최종 결과 경로 요약
# ============================================================

result_paths = pd.DataFrame([
    {
        "stage": "01 Raw individual",
        "folder_or_file": str(DIRS["raw_individual"]),
        "description": "원본 CSV 개별 복사본"
    },
    {
        "stage": "01 Raw combined",
        "folder_or_file": str(raw_combined_path),
        "description": "원본 CSV 전체 combined"
    },
    {
        "stage": "02 Preprocessed individual",
        "folder_or_file": str(DIRS["preprocessed_individual"]),
        "description": "전처리 개별 파일"
    },
    {
        "stage": "02 Preprocessed combined",
        "folder_or_file": str(preprocessed_combined_path),
        "description": "전처리 전체 combined"
    },
    {
        "stage": "03 Augmented individual",
        "folder_or_file": str(DIRS["augmented_individual"]),
        "description": "Gaussian Noise 증강 개별 파일"
    },
    {
        "stage": "03 Augmented combined",
        "folder_or_file": str(augmented_combined_path),
        "description": "증강 데이터 전체 combined"
    },
    {
        "stage": "04 Preprocessed + Augmented",
        "folder_or_file": str(preprocessed_plus_augmented_path),
        "description": "전처리 데이터와 증강 데이터 결합"
    },
    {
        "stage": "05 Feature individual",
        "folder_or_file": str(DIRS["feature_individual"]),
        "description": "Window feature 적용 개별 파일"
    },
    {
        "stage": "05 Feature combined",
        "folder_or_file": str(feature_combined_path),
        "description": "Window feature 적용 전체 combined"
    },
    {
        "stage": "06 Sampled individual",
        "folder_or_file": str(DIRS["sampled_individual"]),
        "description": "최종 샘플링 개별 파일"
    },
    {
        "stage": "06 Sampled combined",
        "folder_or_file": str(sampled_combined_path),
        "description": "Azure ML 업로드용 최종 학습 파일"
    },
])

display(result_paths)

print("\nAzure ML에 우선 업로드할 최종 파일:")
print(sampled_combined_path)

print("\nTarget column:")
print("current_status_label")

print("\nFeature에서 제외 권장:")
print(["current_status_label", "status_name", "is_risk_zone", "is_near_risk_zone"])

In [ ]:
# ============================================================
# 15. 품질 확인: 증강 전/후 센서 통계 비교
# ============================================================

sensor_cols = ["voltage", "current", "battery_temp", "ambient_temp"]

summary_rows = []

for col in sensor_cols:
    if col in preprocessed_combined_df.columns and col in augmented_combined_df.columns:
        summary_rows.append({
            "column": col,
            "preprocessed_mean": preprocessed_combined_df[col].mean(),
            "augmented_mean": augmented_combined_df[col].mean(),
            "preprocessed_std": preprocessed_combined_df[col].std(),
            "augmented_std": augmented_combined_df[col].std(),
            "mean_diff": augmented_combined_df[col].mean() - preprocessed_combined_df[col].mean(),
            "std_diff": augmented_combined_df[col].std() - preprocessed_combined_df[col].std(),
        })

sensor_compare_df = pd.DataFrame(summary_rows)
display(sensor_compare_df)

print("권장 해석:")
print("- mean_diff가 너무 크면 NOISE_LEVEL이 과할 수 있습니다.")
print("- 라벨 분포가 너무 많이 바뀌면 NOISE_LEVEL을 0.005 정도로 낮춰보세요.")
print("- 거의 변화가 없으면 NOISE_LEVEL을 0.02 정도로 올려볼 수 있습니다.")